# PPO (Proximal Policy Optimization) — Step by Step

**Environment**: CartPole-v1  
**Algorithm**: PPO-Clip with Actor-Critic and GAE

---

## Overview

| Component | Role |
|-----------|------|
| Actor | Outputs action probabilities π(a\|s) — *what to do* |
| Critic | Outputs state value V(s) — *how good is this state* |
| Rollout Buffer | Collects on-policy trajectories before each update |
| GAE | Computes advantage estimates with bias-variance tradeoff |
| PPO-Clip | Prevents overly large policy updates |

**Core Update Rule (Actor):**
$$L^{\text{CLIP}}(\theta) = \mathbb{E}\left[\min\left(r_t(\theta) A_t,\ \text{clip}(r_t(\theta), 1-\epsilon, 1+\epsilon) A_t\right)\right]$$
where $r_t(\theta) = \dfrac{\pi_{\theta}(a_t|s_t)}{\pi_{\theta_{\text{old}}}(a_t|s_t)}$

## 1. Imports & Dependencies

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Categorical
import gymnasium as gym
from collections import deque
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)

print("Gymnasium:", gym.__version__)
print("PyTorch:  ", torch.__version__)

## 2. Hyperparameters

**Key PPO-specific parameters:**

| Parameter | Value | Meaning |
|-----------|-------|---------|
| `CLIP_EPSILON` | 0.2 | Maximum allowed ratio change: [0.8, 1.2] |
| `K_EPOCHS` | 4 | Reuse same rollout data this many times |
| `GAE_LAMBDA` | 0.95 | GAE smoothing: 0=TD(0), 1=Monte Carlo |
| `ROLLOUT_STEPS` | 512 | Collect this many steps before each update |

**DQN vs PPO data efficiency:**
- DQN: each experience used once, then discarded (or stays in buffer)
- PPO: each rollout reused `K_EPOCHS` times, then **discarded** (on-policy)

In [ ]:
# ── Training ──────────────────────────────────────
EPISODES       = 500      # Total episodes
GAMMA          = 0.99     # Discount factor

# ── Actor-Critic ───────────────────────────────────
ACTOR_LR       = 3e-4     # Actor learning rate (smaller — more sensitive)
CRITIC_LR      = 1e-3     # Critic learning rate (larger — value regression)

# ── PPO-Clip ───────────────────────────────────────
CLIP_EPSILON   = 0.2      # Clip ratio to [1-0.2, 1+0.2] = [0.8, 1.2]
K_EPOCHS       = 4        # Number of gradient updates per rollout
ENTROPY_COEF   = 0.01     # Weight for entropy bonus (encourages exploration)

# ── GAE ────────────────────────────────────────────
GAE_LAMBDA     = 0.95     # 0 = pure TD(0), 1 = full Monte Carlo

# ── Rollout ────────────────────────────────────────
ROLLOUT_STEPS  = 512      # Steps to collect before each update

print("Hyperparameters loaded.")
print(f"  Clip range: [{1-CLIP_EPSILON:.1f}, {1+CLIP_EPSILON:.1f}]")
print(f"  K_EPOCHS: {K_EPOCHS}  |  Rollout steps: {ROLLOUT_STEPS}")

## 3. Actor Network (Policy)

The Actor outputs a **probability distribution** over actions.

```
Input(4) → Linear(64) → Tanh → Linear(64) → Tanh → Linear(2) → Softmax
                                                         ↑
                                           [P(left), P(right)]
```

**Why Tanh instead of ReLU?**  
PPO's loss has both positive and negative regions. Tanh outputs are  
bounded in (-1, 1), which gives smoother gradients compared to ReLU's  
hard zero floor — especially important for policy networks.

**Why return a distribution object, not just an action?**  
We need `log_prob` to compute the ratio $r_t = \pi_{new}/\pi_{old}$,  
and `entropy` for the exploration bonus.

In [ ]:
class Actor(nn.Module):
    """
    Policy network: state → action probability distribution
    Returns a Categorical distribution (can sample, compute log_prob, entropy)
    """
    def __init__(self, state_size, action_size):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_size, 64),
            nn.Tanh(),
            nn.Linear(64, 64),
            nn.Tanh(),
            nn.Linear(64, action_size),   # Raw logits (softmax applied inside Categorical)
        )

    def forward(self, x):
        logits = self.net(x)
        # Categorical wraps softmax + sampling + log_prob computation
        return Categorical(logits=logits)


# Sanity check
_actor = Actor(state_size=4, action_size=2)
_s = torch.zeros(1, 4)
_dist = _actor(_s)
_a = _dist.sample()
print("Action sampled:  ", _a.item())
print("Log probability: ", _dist.log_prob(_a).item())
print("Probabilities:   ", _dist.probs.detach().numpy())
print("Entropy:         ", _dist.entropy().item())

## 4. Critic Network (Value Function)

The Critic estimates **V(s)** — the expected total discounted reward from state s.

```
Input(4) → Linear(64) → Tanh → Linear(64) → Tanh → Linear(1)
                                                          ↑
                                               Scalar V(s) estimate
```

**How Critic helps Actor:**  
$$A_t = R_t - V(s_t) \quad \text{(simplified)}$$

- $A_t > 0$ : this action did *better* than expected → increase its probability
- $A_t < 0$ : this action did *worse* than expected → decrease its probability

Without the Critic, we'd only know "did this episode go well overall",  
not "which specific actions were good or bad" (high variance).

In [ ]:
class Critic(nn.Module):
    """
    Value network: state → scalar V(s)
    Used to compute advantage estimates and as the training target for itself.
    """
    def __init__(self, state_size):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_size, 64),
            nn.Tanh(),
            nn.Linear(64, 64),
            nn.Tanh(),
            nn.Linear(64, 1),    # Single scalar output
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)   # (batch, 1) → (batch,)


# Sanity check
_critic = Critic(state_size=4)
_val = _critic(torch.zeros(1, 4))
print("Value estimate (random init):", _val.item())   # Should be some random float

## 5. Rollout Buffer

**Key difference from DQN's ReplayBuffer:**

| | DQN ReplayBuffer | PPO RolloutBuffer |
|---|---|---|
| Sampling | Random (off-policy OK) | Sequential trajectory |
| Reuse | Stored long-term | Cleared after each update |
| Extra fields | — | `log_probs`, `values` |
| Why | Break correlations | Track old policy for ratio |

We store `log_prob` and `value` at collection time (old policy).  
During training, we compute new `log_prob` to get ratio $r_t = \pi_{new}/\pi_{old}$.

In [ ]:
class RolloutBuffer:
    """
    Stores a fixed-length trajectory collected under the CURRENT policy.
    Cleared completely after each update (on-policy requirement).
    """
    def __init__(self):
        self.clear()

    def clear(self):
        self.states    = []
        self.actions   = []
        self.rewards   = []
        self.dones     = []
        self.log_probs = []   # log π_old(a|s) — needed to compute ratio
        self.values    = []   # V(s) at collection time — needed for GAE

    def push(self, state, action, reward, done, log_prob, value):
        self.states.append(state)
        self.actions.append(action)
        self.rewards.append(reward)
        self.dones.append(done)
        self.log_probs.append(log_prob)
        self.values.append(value)

    def get_tensors(self):
        return (
            torch.FloatTensor(np.array(self.states)),
            torch.LongTensor(self.actions),
            torch.FloatTensor(self.rewards),
            torch.FloatTensor(self.dones),
            torch.FloatTensor(self.log_probs),
            torch.FloatTensor(self.values),
        )

    def __len__(self):
        return len(self.states)

print("RolloutBuffer defined.")

## 6. Generalized Advantage Estimation (GAE)

Simple advantage: $A_t = G_t - V(s_t)$ where $G_t = \sum_{k=0}^T \gamma^k r_{t+k}$

Problem: $G_t$ requires the full episode, has high variance.

**GAE** introduces a smoothing parameter $\lambda$ to trade off:

$$\delta_t = r_t + \gamma V(s_{t+1}) - V(s_t) \quad \text{(TD error)}$$
$$A_t^{\text{GAE}} = \delta_t + (\gamma\lambda) \delta_{t+1} + (\gamma\lambda)^2 \delta_{t+2} + \cdots$$

| λ | Bias | Variance | Method |
|---|------|----------|--------|
| 0 | High | Low | Pure TD(0) |
| 1 | Low | High | Monte Carlo |
| **0.95** | **Balanced** | **Balanced** | **Default PPO** |

**Computed recursively backwards** (from last step to first):
$$A_t = \delta_t + \gamma\lambda(1-d_t) A_{t+1}$$

In [ ]:
def compute_gae(rewards, dones, values, last_value, gamma=GAMMA, lam=GAE_LAMBDA):
    """
    Compute GAE advantages and returns for a rollout trajectory.

    Args:
        rewards    : list of rewards [r_0, r_1, ..., r_{T-1}]
        dones      : list of done flags (1.0 if episode ended)
        values     : V(s) estimates from Critic [V_0, ..., V_{T-1}]
        last_value : V(s_T) — value of the state AFTER the last step
                     (0 if that step was terminal)

    Returns:
        advantages : (T,) tensor of GAE advantage estimates
        returns    : (T,) tensor = advantages + values (used to train Critic)
    """
    T = len(rewards)
    advantages = np.zeros(T, dtype=np.float32)
    gae = 0.0

    # Traverse backwards so each step can use the next step's advantage
    for t in reversed(range(T)):
        # V(s_{t+1}): use last_value if this is the final step
        next_val  = values[t + 1] if t < T - 1 else last_value
        # If episode ended here, no future value
        not_done  = 1.0 - dones[t]
        # TD error: actual reward + discounted next value - current estimate
        delta     = rewards[t] + gamma * next_val * not_done - values[t]
        # GAE recursive update
        gae       = delta + gamma * lam * not_done * gae
        advantages[t] = gae

    returns = advantages + np.array(values)

    # Normalise advantages: mean=0, std=1 for stable gradients
    advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)
    return torch.FloatTensor(advantages), torch.FloatTensor(returns)


# Demo with dummy data
_r  = [1.0, 1.0, 1.0, 1.0, 1.0]
_d  = [0.0, 0.0, 0.0, 0.0, 1.0]
_v  = [0.5, 0.6, 0.7, 0.6, 0.5]
_adv, _ret = compute_gae(_r, _d, _v, last_value=0.0)
print("Advantages (normalised):", _adv.numpy().round(3))
print("Returns:                ", _ret.numpy().round(3))

## 7. PPO Agent

**The critical step — PPO-Clip loss:**

```
ratio  = exp(log_π_new - log_π_old)          # π_new / π_old
surr1  = ratio * advantage                    # Unclipped objective
surr2  = clip(ratio, 1-ε, 1+ε) * advantage   # Clipped objective
loss   = -mean(min(surr1, surr2))             # Take the conservative (min) one
```

**Why `min`?**
- If advantage > 0 (good action): we want to increase probability, but not too much → clamp at 1+ε
- If advantage < 0 (bad action): we want to decrease probability, but not too much → clamp at 1-ε
- `min(surr1, surr2)` always picks the more *pessimistic* (conservative) estimate

In [ ]:
class PPOAgent:
    def __init__(self, state_size, action_size):
        self.actor  = Actor(state_size, action_size)
        self.critic = Critic(state_size)

        # Separate optimizers — different learning rates
        self.actor_optimizer  = optim.Adam(self.actor.parameters(),  lr=ACTOR_LR)
        self.critic_optimizer = optim.Adam(self.critic.parameters(), lr=CRITIC_LR)

        self.buffer = RolloutBuffer()

    # ────────────────────────────────────────────────────────
    def act(self, state):
        """
        Sample action from current policy.
        Also returns log_prob and value needed for training later.
        """
        state_t = torch.FloatTensor(state).unsqueeze(0)
        with torch.no_grad():
            dist  = self.actor(state_t)
            value = self.critic(state_t)
        action   = dist.sample()
        log_prob = dist.log_prob(action)
        return action.item(), log_prob.item(), value.item()

    # ────────────────────────────────────────────────────────
    def train(self, last_value):
        """
        Update Actor and Critic using the collected rollout.
        Repeats K_EPOCHS times on the same data.
        """
        states, actions, rewards, dones, old_log_probs, values =             self.buffer.get_tensors()

        # Compute advantages and returns for the entire rollout
        advantages, returns = compute_gae(
            rewards.numpy(), dones.numpy(), values.numpy(), last_value
        )

        actor_loss_log  = []
        critic_loss_log = []

        for epoch in range(K_EPOCHS):

            # ── Actor Loss (PPO-Clip) ───────────────────────────────────
            dist          = self.actor(states)
            new_log_probs = dist.log_prob(actions)       # log π_new(a|s)
            entropy       = dist.entropy().mean()         # H[π] — exploration bonus

            # r_t = π_new / π_old  (in log space for numerical stability)
            ratio = torch.exp(new_log_probs - old_log_probs)

            surr1 = ratio * advantages
            surr2 = torch.clamp(ratio, 1 - CLIP_EPSILON, 1 + CLIP_EPSILON) * advantages
            # Negative because we MAXIMISE the objective but optimizer MINIMISES loss
            actor_loss = -torch.min(surr1, surr2).mean() - ENTROPY_COEF * entropy

            self.actor_optimizer.zero_grad()
            actor_loss.backward()
            nn.utils.clip_grad_norm_(self.actor.parameters(), max_norm=0.5)
            self.actor_optimizer.step()

            # ── Critic Loss (MSE against GAE returns) ───────────────────
            v_pred      = self.critic(states)
            critic_loss = nn.MSELoss()(v_pred, returns)

            self.critic_optimizer.zero_grad()
            critic_loss.backward()
            nn.utils.clip_grad_norm_(self.critic.parameters(), max_norm=0.5)
            self.critic_optimizer.step()

            actor_loss_log.append(actor_loss.item())
            critic_loss_log.append(critic_loss.item())

        # On-policy: discard rollout after update
        self.buffer.clear()
        return np.mean(actor_loss_log), np.mean(critic_loss_log)

print("PPOAgent defined.")

## 8. Training Loop

**PPO's training rhythm:**
```
Collect ROLLOUT_STEPS steps using current policy
    ↓
Compute GAE advantages for the whole rollout
    ↓
Run K_EPOCHS gradient updates on this rollout
    ↓
Clear buffer — on-policy: old data is stale
    ↓
Repeat
```

This is fundamentally different from DQN which trains every single step.

In [ ]:
def train_ppo():
    env   = gym.make("CartPole-v1")
    agent = PPOAgent(
        state_size  = env.observation_space.shape[0],   # 4
        action_size = env.action_space.n,               # 2
    )

    scores     = []
    recent_100 = deque(maxlen=100)
    step_count = 0

    print("=" * 60)
    print("PPO Training — CartPole-v1")
    print("=" * 60)

    for episode in range(1, EPISODES + 1):
        state, _ = env.reset()
        total_reward = 0

        while True:
            # Act: sample from policy, collect log_prob and value
            action, log_prob, value = agent.act(state)

            next_state, reward, term, trunc, _ = env.step(action)
            done = term or trunc

            # Store full transition including log_prob and value
            agent.buffer.push(state, action, reward, float(done), log_prob, value)

            state        = next_state
            total_reward += reward
            step_count   += 1

            # Trigger update when rollout buffer is full
            if step_count % ROLLOUT_STEPS == 0:
                with torch.no_grad():
                    last_val = agent.critic(
                        torch.FloatTensor(state).unsqueeze(0)
                    ).item()
                # If episode just ended, no bootstrap value
                agent.train(last_val if not done else 0.0)

            if done:
                break

        scores.append(total_reward)
        recent_100.append(total_reward)

        if episode % 20 == 0:
            avg = np.mean(recent_100)
            print(f"  Ep {episode:4d} | Avg(100): {avg:6.1f} | Steps: {step_count:6d}")

        if len(recent_100) == 100 and np.mean(recent_100) >= 475:
            print(f"\n✓ Solved at episode {episode}! Avg(100): {np.mean(recent_100):.1f}")
            break

    env.close()
    return scores, agent

ppo_scores, ppo_agent = train_ppo()

## 9. Results

In [ ]:
plot_scores(ppo_scores, "PPO — CartPole-v1 Training Curve", color='darkorange')

print(f"Best episode:           {max(ppo_scores):.0f}")
print(f"Final avg (last 100):   {np.mean(ppo_scores[-100:]):.1f}")

## 10. DQN vs PPO — Comparison

In [ ]:
# Run this cell only after running both DQN and PPO notebooks
# (or rerun dqn training above if needed)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, scores, label, color in zip(
    axes,
    [dqn_scores, ppo_scores],
    ["DQN", "PPO"],
    ["steelblue", "darkorange"]
):
    w = 20
    rolling = [np.mean(scores[max(0,i-w):i+1]) for i in range(len(scores))]
    ax.plot(scores,  alpha=0.25, color=color)
    ax.plot(rolling, color=color, linewidth=2)
    ax.axhline(475, color='red', linestyle='--', linewidth=1)
    ax.set_title(label, fontsize=14)
    ax.set_xlabel("Episode")
    ax.set_ylabel("Total reward")
    ax.set_ylim(0, 520)
    ax.grid(alpha=0.3)

plt.suptitle("DQN vs PPO — CartPole-v1", fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

print("=" * 50)
print(f"DQN  final avg (last 100): {np.mean(dqn_scores[-100:]):.1f}")
print(f"PPO  final avg (last 100): {np.mean(ppo_scores[-100:]):.1f}")
print()
print("Key differences observed:")
print("  DQN: Learns faster early (off-policy, reuses all experience)")
print("  PPO: More stable convergence (clip prevents catastrophic updates)")

## 11. Algorithm Summary

### DQN

```
Initialize Q, Q_target with same random weights
Initialize ReplayBuffer

For each episode:
    For each step:
        ε-greedy action selection
        Store (s, a, r, s', done) in Buffer
        If Buffer ≥ batch_size:
            Sample 64 random experiences
            Compute targets: y = r + γ max Q_target(s', a')
            Update Q to minimise MSE(Q(s,a), y)
    Decay ε
    Every 10 episodes: Q_target ← Q
```

### PPO

```
Initialize Actor, Critic with random weights

For each episode:
    For each step:
        Sample action from Actor distribution
        Store (s, a, r, done, log_prob, value) in RolloutBuffer
        Every 512 steps:
            Compute GAE advantages
            For K=4 epochs:
                ratio = exp(log_π_new - log_π_old)
                actor_loss  = -min(ratio×A, clip(ratio, 0.8, 1.2)×A)
                critic_loss = MSE(V_pred, returns)
            Clear RolloutBuffer
```

### When to use which?

| Scenario | Prefer |
|----------|--------|
| Discrete actions, large replay possible | DQN |
| Continuous action spaces | PPO |
| Need sample efficiency | DQN |
| Need stability, simpler tuning | PPO |
| Simulated environments (fast data) | PPO |